# 02 - LLM-as-judge baseline (Phase 3), GPU

Scores text-only LLM judges on **exactly the same held-out rows** as the probe (hops 1-2 of the canonical test split,
1,628 rows) and prints the probe's numbers on those same rows next to them.

* **Accelerator: GPU T4 x2** (7B/8B models are split across both GPUs).
* **Internet: On.**  **Secret:** `HF_TOKEN` (Add-ons -> Secrets) from an account that accepted the Llama 3.2 and Llama 3.1 licences.
* **Input:** the Kaggle Dataset made from `linguafranca_bundle.zip` (already contains the probe predictions).
* **Runtime:** ~1 h for all four models including downloads. No hidden states needed.

Output: `/kaggle/working/llm_judge_results.zip` -> `summary.md` (probe vs judges), `summary.json`, per-row CSVs.

In [ ]:
# Find the uploaded bundle (any Kaggle Dataset containing lf_common.py) and work on a writable copy.
import os, sys, shutil
from pathlib import Path
hits = sorted(Path("/kaggle/input").rglob("lf_common.py"))
assert hits, ("Bundle not found. Click 'Add Input' (right panel) and add the Kaggle Dataset you created "
              "from linguafranca_bundle.zip.")
src = hits[0].parent
WORK = Path("/kaggle/working/lf")
if not WORK.exists():
    shutil.copytree(src, WORK)
os.chdir(WORK); sys.path.insert(0, str(WORK))
print("bundle:", src, "->", WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# Hugging Face login (needed for the gated Llama models).
# Kaggle: Add-ons -> Secrets -> add a secret named HF_TOKEN (token from an account that has accepted
# the Llama 3.2 / Llama 3.1 licences on huggingface.co) and switch it on for this notebook.
import os
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN); os.environ["HF_TOKEN"] = HF_TOKEN
    print("Logged in to Hugging Face.")
else:
    print("WARNING: no HF_TOKEN secret - gated Llama models will fail to download (Qwen still works).")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
MODELS = "meta-llama/Llama-3.2-3B-Instruct,Qwen/Qwen2.5-3B-Instruct,Qwen/Qwen2.5-7B-Instruct,meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
!python evaluate_llm_baseline.py --data data/canonical/augmented_v2.jsonl --models "{MODELS}" --probe-predictions outputs/probing/test_predictions.csv --out outputs/llm_judge

In [ ]:
print(open("outputs/llm_judge/summary.md").read())

In [ ]:
# Package the results; download the zip from the Output section (right panel) of this notebook.
import shutil
shutil.make_archive("/kaggle/working/llm_judge_results", "zip", "/kaggle/working/lf/outputs/llm_judge")
print("Download /kaggle/working/llm_judge_results.zip from the Output tab.")